# webbot SFT(検索→読む→答える、Google Colab)

事前学習したモデルに、「質問→検索語→検索した段落→答え」の並びを教えます(`pretrain_colab.ipynb` の後に実行します)。上から順に実行してください。

- ランタイムは **T4 GPU** にします
- 事前学習のチェックポイント(`MyDrive/webbot/runs/base/checkpoint.pt`)とトークナイザー(`MyDrive/webbot/data/tokenizer.json`)を使います
- 検索の対象は jawiki の最後のシャード(6.4万記事)、質問は auto-wiki-qa のうちその記事についてのもの(約10.6万件)です
- **接続が切れたら**: もう一度上から順に実行してください。データの作成は済んでいれば飛ばし、学習は続きから再開します

## 1. 設定

In [ ]:
STEPS = 6000                  # 1ステップ32問(学習用データ約8.6万問を約2周)
BATCH, GRAD_ACCUM = 32, 4      # 32問を8問ずつ4回に分けて計算(長い問が集まったバッチでもT4に収めるため。結果は同じ)
LR, WARMUP = 3e-4, 100
EVAL_INTERVAL = 500
TEST_EXAMPLES = 500           # 最後に答えさせる評価用の質問の数

DRIVE = "/content/drive/MyDrive/webbot"
BASE_CKPT = f"{DRIVE}/runs/base/checkpoint.pt"
TOKENIZER = f"{DRIVE}/data/tokenizer.json"
SFT_DRIVE = f"{DRIVE}/sft"    # train/val/test.jsonl
RUN_DRIVE = f"{DRIVE}/runs/sft"

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 2. GPUとドライブ

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")
assert os.path.exists(BASE_CKPT), "事前学習のチェックポイントが見つかりません(pretrain_colab.ipynb を先に実行してください)"

## 3. コードの取得とインストール

In [ ]:
if os.path.exists("/content/webbot"):
    !git -C /content/webbot pull -q
else:
    !git clone -q https://github.com/KoRo2222/tiny-gpt-librarian /content/webbot
%cd /content/webbot
!pip -q install pyarrow maturin
if not os.path.exists("/root/.cargo/bin/cargo"):
    !curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -q
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
!pip -q install ./rust

## 4. SFTデータの作成(初回だけ)

検索の対象にするシャードと質問のファイルをダウンロードし、BM25の索引を作って、質問ごとに上位3件の段落を引きます。答えがその段落に含まれる質問だけを使います。2コアのColabで十数分〜30分の見込みです。

In [ ]:
from pathlib import Path
if (Path(SFT_DRIVE) / "test.jsonl").exists():
    print("SFT data found in", SFT_DRIVE)
else:
    from huggingface_hub import hf_hub_download
    for repo, name, dest in [
        ("hpprc/jawiki", "data/train-00021-of-00022.parquet", "/content/sft_raw/jawiki"),
        ("cl-nagoya/auto-wiki-qa", "data/train-00003-of-00004.parquet", "/content/sft_raw/qa"),
    ]:
        path = hf_hub_download(repo, name, repo_type="dataset", local_dir="/content/sft_raw/hf")
        os.makedirs(dest, exist_ok=True)
        os.replace(path, f"{dest}/{Path(name).name}")
    !python scripts/build_index.py --raw /content/sft_raw/jawiki --out /content/index
    !python scripts/make_sft_data.py --index /content/index --qa /content/sft_raw/qa --out {SFT_DRIVE} --workers 2
!wc -l {SFT_DRIVE}/*.jsonl

## 5. SFT

途中で接続が切れたら、1→2→3→5 の順に実行し直すと続きから再開します。最後に、評価用の質問(事前学習でもSFTでも見ていない記事についての質問)に、検索した段落を与えて答えさせ、完全一致の割合を表示します。

GPUのメモリが足りない(`OutOfMemoryError`)ときは、設定の `GRAD_ACCUM` を8にして 1→2→3→5 を実行し直してください(続きから再開し、結果は変わりません)。

In [ ]:
cmd = (f"python scripts/sft.py --init {BASE_CKPT} --tokenizer {TOKENIZER} --data {SFT_DRIVE} "
       f"--out {RUN_DRIVE} --resume --steps {STEPS} --batch-size {BATCH} --grad-accum {GRAD_ACCUM} "
       f"--lr {LR} --warmup {WARMUP} --eval-interval {EVAL_INTERVAL} --test-examples {TEST_EXAMPLES}")
!{cmd}

## 6. 結果

`MyDrive/webbot/runs/sft/train.log` に損失の推移、答えの例、完全一致の割合があります。

In [ ]:
!tail -n 15 {RUN_DRIVE}/train.log